In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import re

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("j-hartmann/emotion-english-distilroberta-base")
tokenizer = AutoTokenizer.from_pretrained("j-hartmann/emotion-english-distilroberta-base")
classifier = pipeline('text-classification', model=model, tokenizer=tokenizer)
classifier("Wow! I didn't expect it", top_k=None)

C:\Users\gergi_6afqda2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[{'label': 'surprise', 'score': 0.9813257455825806},
 {'label': 'neutral', 'score': 0.004848384764045477},
 {'label': 'anger', 'score': 0.003918680362403393},
 {'label': 'joy', 'score': 0.0035004562232643366},
 {'label': 'fear', 'score': 0.002781966235488653},
 {'label': 'disgust', 'score': 0.0021905985195189714},
 {'label': 'sadness', 'score': 0.0014341765781864524}]

**Обрав трансформер класифікації тексту за емоціями (в даному випадку коментарі). Розпізнаються 7 класів: anger, disgust, fear, joy, neutral, sadness, surprise. [Посилання на сторінку в HuggingFace](https://huggingface.co/j-hartmann/emotion-english-distilroberta-base)**

In [ ]:
data = pd.read_parquet('emotion_dataset.parquet')
data.head()

,text,labels,id
0,I’m really sorry about your situation :( Altho...,[25],eecwqtt
1,It's wonderful because it's awful. At not with.,[0],ed5f85d
2,"Kings fan here, good luck to you guys! Will be...",[13],een27c3
3,"I didn't know that, thank you for teaching me ...",[15],eelgwd1
4,They got bored from haunting earth for thousan...,[27],eem5uti


**Датасет знайшов з тих, на яких був натренований трансформер. [Посилання](https://huggingface.co/datasets/google-research-datasets/go_emotions)**

In [ ]:
def preprocessing_text(texts):
    texts = re.sub(r'<.*?>', '', texts)
    texts = re.sub(r'[^a-zA-Z]', ' ', texts)
    return ' '.join(x.lower() for x in texts.split())

data['text'] = data['text'].apply(lambda x : preprocessing_text(x))
data.head()

,text,labels,id
0,i m really sorry about your situation although...,[25],eecwqtt
1,it s wonderful because it s awful at not with,[0],ed5f85d
2,kings fan here good luck to you guys will be a...,[13],een27c3
3,i didn t know that thank you for teaching me s...,[15],eelgwd1
4,they got bored from haunting earth for thousan...,[27],eem5uti


**Почистив від зайвих симолів текст**

In [ ]:
text = data['text']
for i in range(12):
    print(text[i])
    print(classifier(text[i]))


i m really sorry about your situation although i love the names sapphira cirilla and scarlett
[{'label': 'sadness', 'score': 0.9828733801841736}]
it s wonderful because it s awful at not with
[{'label': 'sadness', 'score': 0.5785370469093323}]
kings fan here good luck to you guys will be an interesting game to watch
[{'label': 'joy', 'score': 0.749116837978363}]
i didn t know that thank you for teaching me something today
[{'label': 'joy', 'score': 0.468694806098938}]
they got bored from haunting earth for thousands of years and ultimately moved on to the afterlife
[{'label': 'fear', 'score': 0.847414493560791}]
thank you for asking questions and recognizing that there may be things that you don t know or understand about police tactics seriously thank you
[{'label': 'neutral', 'score': 0.39485952258110046}]
you re welcome
[{'label': 'neutral', 'score': 0.4785291254520416}]
congrats on your job too
[{'label': 'neutral', 'score': 0.4933770000934601}]
i m sorry to hear that friend it s f

**Перевірка на датасеті, видно, що в цілому все співпадає, є деякі неточності пов'язані з малим набором емоцій**